# HDB Resale Flat ETL

This notebook contains the full ETL flow for HDB resale flat data from January 2012 to December 2016.

In [ ]:
from pathlib import Path
from datetime import date
import re
import requests
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder.appName("HDBResaleETL").master("local[*]").getOrCreate())
spark.sparkContext.setLogLevel("WARN")
RAW_DIR = Path("data/raw")
OUTPUT_DIR = Path("data/output")
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DATE = date.today()
print("Run date:", RUN_DATE)

## 1. Download source files

The files are saved without modification. If the website structure changes, place the CSV files in `data/raw`.

In [ ]:
COLLECTION_URL = "https://data.gov.sg/collections/189/view"

def download_csv_files():
    response = requests.get(COLLECTION_URL, timeout=60)
    response.raise_for_status()
    urls = sorted(set(re.findall(r'https?://[^"\s]+?\.csv(?:\?[^"\s]*)?', response.text)))
    downloaded = []
    for index, url in enumerate(urls, start=1):
        file_path = RAW_DIR / f"hdb_resale_{index}.csv"
        file_response = requests.get(url, timeout=120)
        if file_response.ok and len(file_response.content) > 1000:
            file_path.write_bytes(file_response.content)
            downloaded.append(str(file_path))
    return downloaded

if not list(RAW_DIR.glob("*.csv")):
    try:
        print("Downloaded:", download_csv_files())
    except Exception as exc:
        print("Automatic download did not work:", exc)
        print("Place the source CSV files in data/raw and continue.")
else:
    print("Using existing raw files")

## 2. Read and combine the raw files

In [ ]:
raw_df = (spark.read.option("header", True).option("inferSchema", False).option("mode", "PERMISSIVE").csv(str(RAW_DIR / "*.csv")))
print("Raw rows:", raw_df.count())
print(raw_df.columns)
raw_df.show(5, truncate=False)

## 3. Basic profiling

In [ ]:
profile_parts = []
for c in raw_df.columns:
    profile_parts.append(raw_df.select(
        F.lit(c).alias("column_name"),
        F.count("*").alias("row_count"),
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias("null_or_blank"),
        F.countDistinct(c).alias("distinct_count")
    ))
profile_df = profile_parts[0]
for part in profile_parts[1:]:
    profile_df = profile_df.unionByName(part)
profile_df.show(100, truncate=False)

## 4. Standardise and validate

Allowed category values are taken from the master dataset instead of being hardcoded.

In [ ]:
df = raw_df
expected = ["month","town","flat_type","block","street_name","storey_range","floor_area_sqm","flat_model","lease_commence_date","remaining_lease","resale_price"]
for c in expected:
    if c not in df.columns:
        df = df.withColumn(c, F.lit(None).cast("string"))

df = (df.withColumn("month_date", F.to_date(F.concat(F.col("month"), F.lit("-01"))))
        .withColumn("floor_area_sqm_num", F.col("floor_area_sqm").cast("double"))
        .withColumn("lease_commence_year", F.col("lease_commence_date").cast("int"))
        .withColumn("resale_price_num", F.col("resale_price").cast("double")))

category_columns = ["town", "flat_type", "flat_model", "storey_range"]
allowed = {c: [r[c] for r in df.select(c).where(F.col(c).isNotNull() & (F.trim(F.col(c)) != "")).distinct().collect()] for c in category_columns}

reason = (F.when(F.col("month_date").isNull(), "Invalid month")
 .when(~F.col("month_date").between(F.to_date(F.lit("2012-01-01")), F.to_date(F.lit("2016-12-01"))), "Month outside required period")
 .when(F.col("town").isNull() | (F.trim(F.col("town")) == ""), "Missing town")
 .when(F.col("flat_type").isNull() | (F.trim(F.col("flat_type")) == ""), "Missing flat type")
 .when(F.col("flat_model").isNull() | (F.trim(F.col("flat_model")) == ""), "Missing flat model")
 .when(F.col("storey_range").isNull() | (F.trim(F.col("storey_range")) == ""), "Missing storey range")
 .when(~F.col("town").isin(allowed["town"]), "Unknown town")
 .when(~F.col("flat_type").isin(allowed["flat_type"]), "Unknown flat type")
 .when(~F.col("flat_model").isin(allowed["flat_model"]), "Unknown flat model")
 .when(~F.col("storey_range").isin(allowed["storey_range"]), "Unknown storey range")
 .when(F.col("resale_price_num").isNull() | (F.col("resale_price_num") <= 0), "Invalid resale price")
 .when(F.col("lease_commence_year").isNull() | ~F.col("lease_commence_year").between(1960, RUN_DATE.year), "Invalid lease year"))

validated_df = df.withColumn("failure_reason", reason)
valid_base_df = validated_df.filter(F.col("failure_reason").isNull())
failed_validation_df = validated_df.filter(F.col("failure_reason").isNotNull())
print("Valid:", valid_base_df.count(), "Failed:", failed_validation_df.count())

## 5. Recalculate remaining lease

In [ ]:
lease_end = F.make_date(F.col("lease_commence_year") + F.lit(99), F.lit(1), F.lit(1))
months_left = F.floor(F.months_between(lease_end, F.lit(RUN_DATE.isoformat())))
valid_base_df = (valid_base_df
 .withColumn("remaining_lease_total_months", F.greatest(months_left, F.lit(0)).cast("int"))
 .withColumn("remaining_lease_years", F.floor(F.col("remaining_lease_total_months") / 12).cast("int"))
 .withColumn("remaining_lease_months", (F.col("remaining_lease_total_months") % 12).cast("int"))
 .withColumn("remaining_lease_recalculated", F.concat(F.col("remaining_lease_years"), F.lit(" years "), F.col("remaining_lease_months"), F.lit(" months"))))

## 6. Remove duplicate keys and keep the highest resale price

In [ ]:
key_columns = [c for c in raw_df.columns if c != "resale_price"]
w = Window.partitionBy(*key_columns).orderBy(F.desc("resale_price_num"))
ranked_df = valid_base_df.withColumn("duplicate_rank", F.row_number().over(w))
cleaned_df = ranked_df.filter(F.col("duplicate_rank") == 1).drop("duplicate_rank")
failed_duplicate_df = (ranked_df.filter(F.col("duplicate_rank") > 1).drop("duplicate_rank").withColumn("failure_reason", F.lit("Duplicate key - lower resale price")))

## 7. Mark price anomalies using IQR by town and flat type

In [ ]:
stats = (cleaned_df.groupBy("town", "flat_type")
 .agg(F.expr("percentile_approx(resale_price_num, 0.25)").alias("q1"), F.expr("percentile_approx(resale_price_num, 0.75)").alias("q3"))
 .withColumn("iqr", F.col("q3") - F.col("q1"))
 .withColumn("lower_price_limit", F.col("q1") - F.lit(1.5) * F.col("iqr"))
 .withColumn("upper_price_limit", F.col("q3") + F.lit(1.5) * F.col("iqr")))
cleaned_df = (cleaned_df.join(stats, ["town", "flat_type"], "left")
 .withColumn("is_price_anomaly", (F.col("resale_price_num") < F.col("lower_price_limit")) | (F.col("resale_price_num") > F.col("upper_price_limit"))))
cleaned_df.groupBy("is_price_anomaly").count().show()

## 8. Create and hash the Resale Identifier

In [ ]:
avg_w = Window.partitionBy("month", "town", "flat_type")
transformed_df = (cleaned_df
 .withColumn("average_group_price", F.avg("resale_price_num").over(avg_w))
 .withColumn("block_digits", F.regexp_replace(F.col("block"), "[^0-9]", ""))
 .withColumn("block_part", F.lpad(F.substring(F.col("block_digits"), 1, 3), 3, "0"))
 .withColumn("average_price_part", F.substring(F.floor(F.col("average_group_price")).cast("string"), 1, 2))
 .withColumn("month_part", F.date_format(F.col("month_date"), "MM"))
 .withColumn("town_part", F.substring(F.upper(F.col("town")), 1, 1))
 .withColumn("resale_identifier", F.concat(F.lit("S"), F.col("block_part"), F.col("average_price_part"), F.col("month_part"), F.col("town_part"))))
hashed_df = transformed_df.withColumn("hashed_resale_identifier", F.sha2(F.col("resale_identifier"), 256)).drop("resale_identifier")
transformed_df.select("block", "month", "town", "resale_identifier").show(10, truncate=False)

## 9. Write outputs

In [ ]:
failed_df = failed_validation_df.unionByName(failed_duplicate_df, allowMissingColumns=True)

def write_output(dataframe, folder):
    path = OUTPUT_DIR / folder
    dataframe.write.mode("overwrite").parquet(str(path))
    print("Written:", path)

write_output(cleaned_df, "cleaned")
write_output(transformed_df, "transformed")
write_output(failed_df, "failed")
write_output(hashed_df, "hashed")

## 10. Final checks

In [ ]:
print("Raw:", raw_df.count())
print("Cleaned:", cleaned_df.count())
print("Failed:", failed_df.count())
print("Transformed:", transformed_df.count())
print("Hashed:", hashed_df.count())
print("Duplicate hashes:", hashed_df.groupBy("hashed_resale_identifier").count().filter(F.col("count") > 1).count())
spark.stop()